# RL-Based Adversarial Attack Transferability (Pretrained Models)
**Train DQN to perturb CIFAR-10 images (224×224), test if attack transfers from ResNet-18 → ResNet-34 → ViT-B/16.**
All classifiers use pretrained ImageNet weights — no training needed.
Run with Runtime → Change runtime type → **T4 GPU**

In [ ]:
!pip install torch torchvision tqdm --quiet

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as T
import torchvision.models as M
from torch.utils.data import DataLoader, Subset
import numpy as np, random, os, time, warnings
from collections import deque
import tqdm.notebook as tqdm

warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

IMG_SIZE = 224
transform = T.Compose([
    T.Resize(IMG_SIZE),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## 1. Load CIFAR-10 (resized to 224×224)

In [ ]:
full_trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
full_testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader  = DataLoader(full_testset, batch_size=128, shuffle=False, num_workers=2)

rl_trainset = Subset(full_trainset, list(range(5000)))
rl_testset  = Subset(full_testset, list(range(1000)))
print(f'Train: {len(full_trainset)}  Test: {len(full_testset)}  RL-subset: {len(rl_trainset)}')

## 2. Load Pretrained Classifiers

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total = correct = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

def freeze_backbone(model, head_name):
    for name, p in model.named_parameters():
        if head_name not in name:
            p.requires_grad = False

def train_head(model, loader, epochs=3, lr=1e-2):
    model.train()
    opt = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        total = correct = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            crit(model(x), y).backward()
            opt.step()
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
        print(f'    Ep {ep+1}/{epochs} train acc: {correct/total:.1%}')

train_loader_small = DataLoader(rl_trainset, batch_size=128, shuffle=True, num_workers=2)
small_test = Subset(full_testset, list(range(1000)))
small_loader = DataLoader(small_test, batch_size=128, shuffle=False)

print('\nLoading and calibrating ResNet-18 (surrogate)...')
surrogate = M.resnet18(weights='IMAGENET1K_V1')
surrogate.fc = nn.Linear(512, 10)
surrogate = surrogate.to(device)
freeze_backbone(surrogate, 'fc')
train_head(surrogate, train_loader_small, epochs=3)
print(f'  ResNet-18 test acc: {evaluate(surrogate, small_loader):.1%}')

print('\nLoading and calibrating ResNet-34...')
resnet34 = M.resnet34(weights='IMAGENET1K_V1')
resnet34.fc = nn.Linear(512, 10)
resnet34 = resnet34.to(device)
freeze_backbone(resnet34, 'fc')
train_head(resnet34, train_loader_small, epochs=3)
print(f'  ResNet-34 test acc: {evaluate(resnet34, small_loader):.1%}')

print('\nLoading and calibrating ViT-B/16...')
vit = M.vit_b_16(weights='IMAGENET1K_V1')
vit.heads.head = nn.Linear(768, 10)
vit = vit.to(device)
freeze_backbone(vit, 'heads')
train_head(vit, train_loader_small, epochs=3)
print(f'  ViT-B/16 test acc: {evaluate(vit, small_loader):.1%}')

print('\nAll models calibrated to ~65-75% — ready.')

## 3. RL Attack Environment
- Image: 224×224, patch grid 8×8 = 64 patches (28×28 px each)
- State: 64 noise levels + confidence + step remaining (dim=66)
- Actions: 64 (pick patch to perturb)
- Reward: +15 on misclassification −0.3 per patch −0.1×confidence

In [ ]:
PATCH_SIZE = 28
GRID_SIZE = 8
N_PATCHES = 64

class ImageAttackEnv:
    def __init__(self, classifier, dataset, max_steps=20, noise_scale=0.08):
        self.classifier = classifier; self.classifier.eval()
        self.dataset = dataset
        self.max_steps = max_steps; self.noise_scale = noise_scale
        self.action_space = N_PATCHES

        self.patch_coords = []
        for i in range(GRID_SIZE):
            for j in range(GRID_SIZE):
                self.patch_coords.append((i * PATCH_SIZE, (i + 1) * PATCH_SIZE,
                                          j * PATCH_SIZE, (j + 1) * PATCH_SIZE))
        self.reset()

    def _get_random_image(self):
        idx = np.random.randint(len(self.dataset))
        img, label = self.dataset[idx]
        return img.to(device), torch.tensor([label]).to(device)

    def reset(self):
        self.img, self.label = self._get_random_image()
        self.orig_img = self.img.clone()
        self.noise = torch.zeros(3, IMG_SIZE, IMG_SIZE, device=device)
        self.patches_touched = np.zeros(N_PATCHES, dtype=np.float32)
        self.steps = 0; self.done = False
        return self._get_state()

    def _get_state(self):
        with torch.no_grad():
            logits = self.classifier(self.img.unsqueeze(0))
            probs = torch.softmax(logits, 1)
            confidence = probs[0, self.label].item()
        return np.concatenate([
            self.patches_touched,
            [confidence],
            [1.0 - self.steps / self.max_steps],
        ]).astype(np.float32)

    def _apply_noise(self, patch_idx):
        y0, y1, x0, x1 = self.patch_coords[patch_idx]
        patch = torch.randn(3, PATCH_SIZE, PATCH_SIZE, device=device) * self.noise_scale
        self.noise[:, y0:y1, x0:x1] += patch
        self.img = torch.clamp(self.orig_img + self.noise, -3.0, 3.0)

    def step(self, action):
        if self.done:
            return self._get_state(), 0.0, True, {}
        self._apply_noise(action)
        self.patches_touched[action] = 1.0
        self.steps += 1

        with torch.no_grad():
            logits = self.classifier(self.img.unsqueeze(0))
            pred = logits.argmax(1).item()
            probs = torch.softmax(logits, 1)
            confidence = probs[0, self.label].item()

        misclassified = pred != self.label.item()
        num_patches = int(self.patches_touched.sum())

        if misclassified:
            reward = 15.0 - 0.3 * num_patches
            self.done = True
        else:
            reward = -0.1 * num_patches - 0.1 * confidence

        if self.steps >= self.max_steps:
            self.done = True

        info = {
            'misclassified': misclassified, 'predicted': pred,
            'true_label': self.label.item(), 'patches_used': num_patches,
            'confidence': confidence,
        }
        return self._get_state(), reward, self.done, info

## 4. DQN Agent

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )
    def forward(self, x): return self.net(x)

class ReplayBuffer:
    def __init__(self, cap=50000): self.buf = deque(maxlen=cap)
    def push(self, s, a, r, ns, d): self.buf.append((s, a, r, ns, d))
    def sample(self, bs):
        batch = random.sample(self.buf, min(bs, len(self.buf)))
        s, a, r, ns, d = zip(*batch)
        return (torch.FloatTensor(np.array(s)).to(device),
                torch.LongTensor(a).unsqueeze(1).to(device),
                torch.FloatTensor(r).unsqueeze(1).to(device),
                torch.FloatTensor(np.array(ns)).to(device),
                torch.FloatTensor(d).unsqueeze(1).to(device))
    def __len__(self): return len(self.buf)

class DQNAgent:
    def __init__(self, state_dim, action_dim):
        self.act_dim = action_dim
        self.gamma = 0.99; self.eps = 1.0; self.eps_min = 0.05; self.decay = 0.993
        self.bs = 128; self.target_update = 50; self.steps = 0
        self.q = QNetwork(state_dim, action_dim).to(device)
        self.target = QNetwork(state_dim, action_dim).to(device)
        self.target.load_state_dict(self.q.state_dict()); self.target.eval()
        self.opt = optim.Adam(self.q.parameters(), lr=5e-4)
        self.mem = ReplayBuffer(50000)
    def act(self, state, eval=False):
        if not eval and random.random() < self.eps:
            return random.randint(0, self.act_dim - 1)
        with torch.no_grad():
            return self.q(torch.FloatTensor(state).unsqueeze(0).to(device)).argmax().item()
    def decay_epsilon(self):
        self.eps = max(self.eps_min, self.eps * self.decay)
    def learn(self):
        if len(self.mem) < self.bs: return None
        s, a, r, ns, d = self.mem.sample(self.bs)
        with torch.no_grad():
            q_next = self.target(ns).max(1, keepdim=True)[0]
        q_tgt = r + self.gamma * q_next * (1 - d)
        q_cur = self.q(s).gather(1, a)
        loss = nn.MSELoss()(q_cur, q_tgt)
        self.opt.zero_grad(); loss.backward(); self.opt.step()
        self.steps += 1
        if self.steps % self.target_update == 0:
            self.target.load_state_dict(self.q.state_dict())
        return loss.item()

## 5. Train DQN on ResNet-18 (Surrogate)

In [ ]:
env = ImageAttackEnv(surrogate, rl_trainset)
agent = DQNAgent(state_dim=66, action_dim=N_PATCHES)

N_EPISODES = 600
RENDER_EVERY = 30
best_reward = -999
rewards = []; successes = []

print(f'Training for {N_EPISODES} episodes...')
start = time.time()

for ep in range(1, N_EPISODES + 1):
    state = env.reset()
    ep_r = 0.0; done = False; success = False

    while not done:
        a = agent.act(state)
        ns, r, done, info = env.step(a)
        agent.mem.push(state, a, r, ns, done)
        agent.learn()
        state = ns; ep_r += r
        if info['misclassified']: success = True

    agent.decay_epsilon()
    rewards.append(ep_r)
    successes.append(1 if success else 0)

    if ep_r > best_reward:
        best_reward = ep_r
        torch.save(agent.q.state_dict(), '/content/dqn_best.pt')

    if ep % RENDER_EVERY == 0:
        avg_r = np.mean(rewards[-RENDER_EVERY:])
        avg_s = np.mean(successes[-RENDER_EVERY:])
        elapsed = time.time() - start
        print(f'Ep {ep:3d}/{N_EPISODES} | R: {avg_r:+.1f} | Succ: {avg_s:.0%} | '
              f'Eps: {agent.eps:.3f} | Patches: {info["patches_used"]} | {elapsed:.0f}s')

print(f'\nDone. Best reward: {best_reward:+.1f}')
print(f'Final success rate (last 50): {np.mean(successes[-50:]):.0%}')

## 6. Test Transferability

In [ ]:
agent.q.load_state_dict(torch.load('/content/dqn_best.pt', map_location=device))

@torch.no_grad()
def run_policy(agent, img, label, classifier, max_steps=20):
    noise = torch.zeros(3, IMG_SIZE, IMG_SIZE, device=device)
    patches_touched = np.zeros(N_PATCHES, dtype=np.float32)
    orig = img.clone()
    for step in range(max_steps):
        logits = classifier((orig + noise).unsqueeze(0))
        probs = torch.softmax(logits, 1)
        confidence = probs[0, label].item()
        state = np.concatenate([patches_touched, [confidence],
                                [1.0 - step / max_steps]]).astype(np.float32)
        action = agent.act(state, eval=True)
        y0 = (action // GRID_SIZE) * PATCH_SIZE
        x0 = (action % GRID_SIZE) * PATCH_SIZE
        noise[:, y0:y0+PATCH_SIZE, x0:x0+PATCH_SIZE] += torch.randn(3, PATCH_SIZE, PATCH_SIZE, device=device) * 0.08
        patches_touched[action] = 1.0
        if logits.argmax(1).item() != label:
            return True, step + 1
    return False, 20

def batch_test(agent, model, name, n_samples=500):
    successes = []; queries = []
    for i in tqdm.trange(n_samples, desc=name):
        img, label = rl_testset[i]
        succ, nq = run_policy(agent, img.to(device), label, model)
        successes.append(succ); queries.append(nq)
    return {'name': name, 'success_rate': np.mean(successes), 'avg_queries': np.mean(queries)}

results = [
    batch_test(agent, surrogate, 'ResNet-18 (surrogate)'),
    batch_test(agent, resnet34,   'ResNet-34 (transfer)'),
    batch_test(agent, vit,         'ViT-B/16 (transfer)'),
]

print(f"\n{'='*55}")
print(f"{'Model':>22s} | {'Success':>8s} | {'Avg Queries':>11s}")
print(f"{'-'*55}")
for r in results:
    print(f"{r['name']:>22s} | {r['success_rate']:.0%}     | {r['avg_queries']:.1f}")
print(f"{'='*55}")

## 7. Ablation: Query Budget Effect

In [ ]:
@torch.no_grad()
def run_policy_budget(agent, img, label, classifier, max_steps):
    noise = torch.zeros(3, IMG_SIZE, IMG_SIZE, device=device)
    patches_touched = np.zeros(N_PATCHES, dtype=np.float32)
    orig = img.clone()
    for step in range(max_steps):
        logits = classifier((orig + noise).unsqueeze(0))
        probs = torch.softmax(logits, 1)
        confidence = probs[0, label].item()
        state = np.concatenate([patches_touched, [confidence],
                                [1.0 - step / max_steps]]).astype(np.float32)
        action = agent.act(state, eval=True)
        y0 = (action // GRID_SIZE) * PATCH_SIZE
        x0 = (action % GRID_SIZE) * PATCH_SIZE
        noise[:, y0:y0+PATCH_SIZE, x0:x0+PATCH_SIZE] += torch.randn(3, PATCH_SIZE, PATCH_SIZE, device=device) * 0.08
        patches_touched[action] = 1.0
        if logits.argmax(1).item() != label:
            return True
    return False

budgets = [5, 10, 20, 30]
print(f"{'Budget':>8s} | {'ResNet-18':>10s} | {'ResNet-34':>10s} | {'ViT-B/16':>10s}")
print(f"{'-'*50}")
for b in budgets:
    row = [str(b)]
    for m in [surrogate, resnet34, vit]:
        succ = [run_policy_budget(agent, rl_testset[i][0].to(device), rl_testset[i][1], m, b)
                for i in range(300)]
        row.append(f"{np.mean(succ):.0%}")
    print(f"{row[0]:>8s} | {row[1]:>10s} | {row[2]:>10s} | {row[3]:>10s}")